# Module 04, Unit 03 — Constrained Optimization & Lagrange Multipliers

> **Threat Surfaces: Multivariable Calculus for AI Security**
> fischer³ Education | Module 04 | Unit 03

| | |
|---|---|
| **Estimated time** | 60–70 min |
| **Exercises** | Download PDF from the course repository |
| **Statistical bridges** | Maximum entropy distribution `[MLE]` · Lasso as a constrained problem `[GLM]` |
| **Prerequisite** | Module 04, Units 01 & 02 |

---

## Learning Objectives

By the end of this unit you will be able to:

- [ ] State the equality-constrained optimization problem and explain why unconstrained methods fail
- [ ] Construct the Lagrangian and state the geometric condition $\nabla f = \lambda \nabla g$ at a constrained optimum
- [ ] Apply the method of Lagrange multipliers to solve equality-constrained problems
- [ ] State the KKT conditions for inequality-constrained problems and interpret complementary slackness
- [ ] Derive the maximum entropy discrete distribution over $n$ outcomes subject to a normalization constraint
- [ ] Formulate Lasso regression as a constrained problem and explain geometrically why it produces sparse solutions

---

## Why This Unit

Units 01 and 02 handled unconstrained optimization — find where $\nabla f = \boldsymbol{0}$. But many of the most important problems in statistics and machine learning are **constrained**: we want to minimize a loss subject to a budget on the parameters, or find a distribution that maximizes entropy subject to known expectations.

The method of Lagrange multipliers converts a constrained problem into an unconstrained one by absorbing the constraints into a new function — the **Lagrangian**. The KKT conditions extend this to inequality constraints and are the foundation of convex optimization theory that underlies support vector machines, compressed sensing, and optimal transport.

---

## 1. Equality-Constrained Optimization

**Problem setup.** Minimize (or maximize) $f(\boldsymbol{x})$ subject to $g(\boldsymbol{x}) = 0$, where $f, g : \mathbb{R}^n \to \mathbb{R}$.

The feasible set is the level surface $\{\boldsymbol{x} : g(\boldsymbol{x}) = 0\}$ — a curve in $\mathbb{R}^2$, a surface in $\mathbb{R}^3$, or a hypersurface in $\mathbb{R}^n$.

**Why $\nabla f = \boldsymbol{0}$ is wrong here.** On the constraint surface, we cannot move freely in all directions — we are restricted to tangent directions of the surface. A constrained optimum need not be an unconstrained critical point: $\nabla f$ may be nonzero there.

### 1.1 Geometric Condition

**Theorem (Lagrange).** At a constrained local extremum $\boldsymbol{x}^*$, the gradient of $f$ must be parallel to the gradient of $g$:

$$
\nabla f(\boldsymbol{x}^*) = \lambda\,\nabla g(\boldsymbol{x}^*)
$$

for some scalar $\lambda \in \mathbb{R}$ called the **Lagrange multiplier**.

**Why.** Any feasible direction $\boldsymbol{v}$ at $\boldsymbol{x}^*$ must be tangent to the constraint surface, meaning $\nabla g(\boldsymbol{x}^*)^{\top}\boldsymbol{v} = 0$. At a constrained optimum, the directional derivative of $f$ along every feasible direction must be zero (otherwise we could improve $f$): $\nabla f(\boldsymbol{x}^*)^{\top}\boldsymbol{v} = 0$. So $\nabla f$ is zero on all directions perpendicular to $\nabla g$ — which means $\nabla f$ must itself be parallel to $\nabla g$.

**Geometric picture**: at the optimum, the level sets of $f$ and the constraint curve are tangent to each other. The gradient $\nabla f$ is perpendicular to its level set; $\nabla g$ is perpendicular to the constraint curve. Tangency means these two perpendiculars are parallel — hence $\nabla f = \lambda \nabla g$.

### 1.2 The Lagrangian

**Definition.** The **Lagrangian** combines objective and constraint into a single function:

$$
\mathcal{L}(\boldsymbol{x}, \lambda) \coloneqq f(\boldsymbol{x}) - \lambda\,g(\boldsymbol{x})
$$

The **Lagrange conditions** (necessary conditions for a constrained extremum) are:

$$
\nabla_{\boldsymbol{x}}\,\mathcal{L} = \nabla f(\boldsymbol{x}) - \lambda\,\nabla g(\boldsymbol{x}) = \boldsymbol{0}
$$
$$
\frac{\partial \mathcal{L}}{\partial \lambda} = -g(\boldsymbol{x}) = 0
$$

The second equation just reinstates the constraint $g(\boldsymbol{x}) = 0$. Solving both simultaneously gives all candidate constrained extrema.

**Worked Example 1.1.** Maximize $f(x,y) = xy$ subject to $g(x,y) = x + y - 4 = 0$.

Lagrangian: $\mathcal{L} = xy - \lambda(x + y - 4)$.

$$
\frac{\partial \mathcal{L}}{\partial x} = y - \lambda = 0 \implies y = \lambda
$$
$$
\frac{\partial \mathcal{L}}{\partial y} = x - \lambda = 0 \implies x = \lambda
$$
$$
\frac{\partial \mathcal{L}}{\partial \lambda} = -(x + y - 4) = 0 \implies x + y = 4
$$

From the first two: $x = y$. Substituting: $2x = 4 \implies x = y = 2$, $\lambda = 2$.

Constrained maximum: $f(2, 2) = 4$. (By AM-GM, $xy \leq \left(\frac{x+y}{2}\right)^2 = 4$. ✓)

**Worked Example 1.2.** Minimize $f(x, y) = x^2 + y^2$ (distance from origin squared) subject to $g(x, y) = x + 2y - 5 = 0$.

$$
\nabla f = \lambda \nabla g: \quad (2x, 2y) = \lambda(1, 2)
$$

So $2x = \lambda$ and $2y = 2\lambda$, giving $y = 2x$. Substituting into the constraint: $x + 2(2x) = 5 \implies x = 1$, $y = 2$, $\lambda = 2$.

Minimum distance from origin to the line $x + 2y = 5$: $\sqrt{1 + 4} = \sqrt{5}$.

---

## 2. Multiple Constraints

For $m$ equality constraints $g_1(\boldsymbol{x}) = 0, \ldots, g_m(\boldsymbol{x}) = 0$ with $m < n$, introduce one multiplier per constraint:

$$
\mathcal{L}(\boldsymbol{x}, \boldsymbol{\lambda}) = f(\boldsymbol{x}) - \sum_{j=1}^m \lambda_j\,g_j(\boldsymbol{x})
$$

The Lagrange conditions are $\nabla_{\boldsymbol{x}}\,\mathcal{L} = \boldsymbol{0}$ and $g_j(\boldsymbol{x}) = 0$ for all $j$.

In matrix form, the condition $\nabla f = \boldsymbol{J}_{\boldsymbol{g}}^{\top}\boldsymbol{\lambda}$ requires $\nabla f$ to lie in the column space of the constraint Jacobian — a direct application of Module 03 Unit 03.

---

## 3. Inequality Constraints and KKT Conditions

Many practical problems involve inequality constraints: $h(\boldsymbol{x}) \leq 0$. The **KKT (Karush-Kuhn-Tucker) conditions** extend Lagrange multipliers to this setting.

**Problem:** Minimize $f(\boldsymbol{x})$ subject to $g_j(\boldsymbol{x}) = 0$ ($j = 1, \ldots, m$) and $h_k(\boldsymbol{x}) \leq 0$ ($k = 1, \ldots, r$).

**Lagrangian:**

$$
\mathcal{L}(\boldsymbol{x}, \boldsymbol{\lambda}, \boldsymbol{\mu}) = f(\boldsymbol{x}) - \sum_j \lambda_j g_j(\boldsymbol{x}) - \sum_k \mu_k h_k(\boldsymbol{x})
$$

**KKT Conditions** (necessary conditions for a local minimum, under mild regularity):

$$
\nabla_{\boldsymbol{x}}\,\mathcal{L} = \boldsymbol{0} \qquad \text{(stationarity)}
$$
$$
g_j(\boldsymbol{x}) = 0 \quad \forall j \qquad \text{(equality feasibility)}
$$
$$
h_k(\boldsymbol{x}) \leq 0 \quad \forall k \qquad \text{(inequality feasibility)}
$$
$$
\mu_k \geq 0 \quad \forall k \qquad \text{(dual feasibility)}
$$
$$
\mu_k\,h_k(\boldsymbol{x}) = 0 \quad \forall k \qquad \text{(complementary slackness)}
$$

**Complementary slackness** is the key condition: for each inequality constraint, either the constraint is **active** ($h_k(\boldsymbol{x}) = 0$, constraint is binding, $\mu_k \geq 0$) or it is **inactive** ($h_k(\boldsymbol{x}) < 0$, constraint is slack, $\mu_k = 0$). At most one of $\mu_k$ and $h_k$ can be nonzero — they cannot both be nonzero simultaneously.

**Sufficiency.** For convex $f$ and convex feasible sets (convex $h_k$, affine $g_j$), the KKT conditions are both necessary and sufficient for a global minimum.

---

## 4. Statistical Bridge — Maximum Entropy Distribution `[MLE]`

The **maximum entropy principle** asks: given only that probabilities sum to one, which discrete distribution is least informative — spread as uniformly as possible over $n$ outcomes?

### 4.1 Problem

Maximize the **Shannon entropy**:

$$
H(\boldsymbol{p}) = -\sum_{i=1}^n p_i \log p_i
$$

subject to the normalization constraint:

$$
g(\boldsymbol{p}) = \sum_{i=1}^n p_i - 1 = 0
$$

### 4.2 Lagrangian and Solution

$$
\mathcal{L}(\boldsymbol{p}, \lambda) = -\sum_{i=1}^n p_i \log p_i - \lambda\left(\sum_{i=1}^n p_i - 1\right)
$$

Differentiating with respect to $p_i$:

$$
\frac{\partial \mathcal{L}}{\partial p_i} = -(\log p_i + 1) - \lambda = 0 \implies \log p_i = -(1 + \lambda)
$$

This holds for all $i$ with the same right-hand side — so all $p_i$ are equal. The normalization constraint then gives:

$$
p_i^* = \frac{1}{n} \quad \forall\, i \qquad \text{(uniform distribution)}
$$

The maximum entropy distribution subject only to normalization is the **uniform distribution**. This is not obvious without calculus — it requires the Lagrange method to prove rigorously.

> **Extension.** If we add a constraint on the mean $\sum_i p_i x_i = \mu$ (one more equality constraint, one more multiplier), the maximum entropy distribution becomes the **exponential distribution** for $x_i \geq 0$, or the **Gaussian** for $x_i \in \mathbb{R}$ with a variance constraint. The Gaussian is the maximum entropy distribution for a given mean and variance — a profound result that partially explains its ubiquity in nature and statistics. We return to this in Module 06.

---

## 5. Statistical Bridge — Lasso as Constrained Optimization `[GLM]`

### 5.1 The Constrained Formulation

Lasso regression minimizes the residual sum of squares subject to a budget on the $\ell_1$ norm of the coefficients:

$$
\min_{\boldsymbol{\beta}} \|\boldsymbol{y} - \boldsymbol{X}\boldsymbol{\beta}\|^2 \quad \text{subject to} \quad \|\boldsymbol{\beta}\|_1 \leq t
$$

This is equivalent (via a Lagrange dual argument) to the penalized form:

$$
\min_{\boldsymbol{\beta}} \|\boldsymbol{y} - \boldsymbol{X}\boldsymbol{\beta}\|^2 + \lambda\|\boldsymbol{\beta}\|_1
$$

where $\lambda$ and $t$ are in one-to-one correspondence via the KKT conditions.

### 5.2 Why Lasso Produces Sparse Solutions — Geometric Proof

Both Ridge and Lasso minimize the same residual sum of squares $\|\boldsymbol{y} - \boldsymbol{X}\boldsymbol{\beta}\|^2$ — which has elliptical level contours centered at the OLS solution $\hat{\boldsymbol{\beta}}_{\text{OLS}}$.

The constraint determines the feasible set:

- **Ridge**: $\|\boldsymbol{\beta}\|_2^2 \leq t^2$ — a **circle** (sphere in higher dimensions). Smooth, no corners.
- **Lasso**: $\|\boldsymbol{\beta}\|_1 \leq t$ — a **diamond** (cross-polytope in higher dimensions). Has corners on the coordinate axes.

The constrained optimum is where the expanding OLS contour **first touches** the constraint set. For the diamond, this contact almost always occurs at a corner — a point where one or more $\beta_j = 0$ exactly. For the circle, contact can occur anywhere on the smooth boundary, so exact zeros are not produced.

**KKT interpretation.** The Lasso KKT conditions for $\hat{\boldsymbol{\beta}}_{\text{Lasso}}$ are:

$$
\frac{\partial}{\partial \beta_j}\|\boldsymbol{y} - \boldsymbol{X}\boldsymbol{\beta}\|^2\bigg|_{\hat{\boldsymbol{\beta}}} = -\lambda\,\text{sign}(\hat{\beta}_j) \quad \text{if } \hat{\beta}_j \neq 0
$$
$$
\left|\frac{\partial}{\partial \beta_j}\|\boldsymbol{y} - \boldsymbol{X}\boldsymbol{\beta}\|^2\bigg|_{\hat{\boldsymbol{\beta}}}\right| \leq \lambda \quad \text{if } \hat{\beta}_j = 0
$$

The second condition is complementary slackness: a coefficient is set to zero when the gradient of the loss at that coordinate is smaller in magnitude than $\lambda$ — the constraint is binding and the solution sits at a corner.

---

## Python: Visualization & Verification

The cells below demonstrate and visualize the concepts above. **Run them in order.**

To run this notebook interactively, click the **rocket icon** at the top of the page and select **Open in Colab**.

In [ ]:
# ============================================================
# Install required libraries
# ============================================================
import subprocess, sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

for pkg in ["numpy", "matplotlib", "sympy", "scipy"]:
    install(pkg)

print("All packages installed.")

In [ ]:
# ============================================================
# Imports and configuration
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import sympy as sp
from sympy import symbols, solve, diff, log, Rational, simplify

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (9, 6),
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'lines.linewidth': 2,
    'figure.dpi': 120,
})

TS_BLUE  = '#1e64b4'
TS_AMBER = '#c87814'
TS_GREEN = '#1e8c50'
TS_GRAY  = '#64646e'
TS_RED   = '#b41e1e'
TS_LIGHT = '#f5f7fa'

print('Setup complete.')

### Section 1 — Symbolic Lagrange Multiplier Solutions

Use SymPy to verify the worked examples and solve a three-variable constrained problem that would be tedious by hand.

In [ ]:
# ============================================================
# Section 1 — Symbolic Lagrange multiplier solutions
# ============================================================
x, y, z, lam, mu = symbols('x y z lambda mu', real=True)

# --- Example 1.1: max xy s.t. x + y = 4 ---
f1 = x * y
g1 = x + y - 4
L1 = f1 - lam * g1
sol1 = solve([diff(L1, x), diff(L1, y), diff(L1, lam)], [x, y, lam])
print('Example 1.1: max xy s.t. x+y=4')
print('  Solution:', sol1)
print('  f at solution:', f1.subs(sol1[0] if isinstance(sol1, list) else sol1))

# --- Example 1.2: min x²+y² s.t. x+2y=5 ---
f2 = x**2 + y**2
g2 = x + 2*y - 5
L2 = f2 - lam * g2
sol2 = solve([diff(L2, x), diff(L2, y), diff(L2, lam)], [x, y, lam])
print('\nExample 1.2: min x²+y² s.t. x+2y=5')
print('  Solution:', sol2)
if sol2:
    s = sol2 if isinstance(sol2, dict) else sol2[0]
    dist = sp.sqrt(f2.subs(s))
    print('  Min distance from origin:', dist, '=', sp.simplify(dist))

# --- Three-variable: min x²+y²+z² s.t. x+y+z=1, x+2y=0 ---
print('\nThree-variable: min x²+y²+z² s.t. x+y+z=1 and x+2y=0')
f3 = x**2 + y**2 + z**2
g3a = x + y + z - 1
g3b = x + 2*y
L3 = f3 - lam*g3a - mu*g3b
sol3 = solve([diff(L3, x), diff(L3, y), diff(L3, z),
              diff(L3, lam), diff(L3, mu)], [x, y, z, lam, mu])
print('  Solution:', sol3)
if sol3:
    s3 = sol3 if isinstance(sol3, dict) else sol3[0]
    print('  Min ‖x‖²:', simplify(f3.subs(s3)))

**What do you see?** SymPy recovers the hand solutions for both worked examples and handles the three-variable problem with two constraints cleanly — the same algebraic method, just with more equations to solve simultaneously. Confirm that the two-constraint solution satisfies both $g_{3a} = 0$ and $g_{3b} = 0$ by substituting back.

### Section 2 — Geometric Visualization of Lagrange Conditions

Plot the level curves of $f$ and the constraint curve $g = 0$. At the constrained optimum, the level curve of $f$ and the constraint curve are tangent — i.e., their gradients are parallel.

In [ ]:
# ============================================================
# Section 2 — Geometric visualization: ∇f ∥ ∇g at the optimum
# ============================================================
# f(x,y) = x² + y²  (concentric circles)
# g(x,y) = x + 2y - 5 = 0  (a line)
# Constrained min at (1, 2) from Example 1.2

xg = np.linspace(-1.5, 5.5, 400)
yg = np.linspace(-1.0, 4.0, 400)
X_, Y_ = np.meshgrid(xg, yg)
F_ = X_**2 + Y_**2

fig, ax = plt.subplots(figsize=(8, 7))

# Level curves of f
levels_f = [1, 2, 3, 5, 7, 9, 11]
cs = ax.contour(X_, Y_, F_, levels=levels_f,
                colors=[TS_BLUE], alpha=0.55, linewidths=1.2)
ax.clabel(cs, inline=True, fontsize=8, fmt='f=%g')

# Constraint line x + 2y = 5
x_line = np.linspace(-1, 5.5, 200)
y_line = (5 - x_line) / 2
ax.plot(x_line, y_line, color=TS_AMBER, lw=2.2, label='Constraint: $x + 2y = 5$')

# Constrained optimum
x_opt, y_opt = 1.0, 2.0
ax.plot(x_opt, y_opt, 'o', color=TS_RED, markersize=12, zorder=8,
        label=f'Constrained min $(1, 2)$, $f = {x_opt**2 + y_opt**2}$')

# Gradient arrows at the optimum
scale = 0.38
grad_f_opt = np.array([2*x_opt, 2*y_opt])   # ∇f = (2x, 2y)
grad_g_opt = np.array([1.0, 2.0])            # ∇g = (1, 2)
lam_val = 2.0                                 # λ from our solution

ax.annotate('', xy=(x_opt + grad_f_opt[0]*scale, y_opt + grad_f_opt[1]*scale),
            xytext=(x_opt, y_opt),
            arrowprops=dict(arrowstyle='->', color=TS_BLUE, lw=2.5))
ax.text(x_opt + grad_f_opt[0]*scale + 0.05,
        y_opt + grad_f_opt[1]*scale + 0.05,
        r'$\nabla f$', color=TS_BLUE, fontsize=13, fontweight='bold')

ax.annotate('', xy=(x_opt + grad_g_opt[0]*scale, y_opt + grad_g_opt[1]*scale),
            xytext=(x_opt, y_opt),
            arrowprops=dict(arrowstyle='->', color=TS_AMBER, lw=2.5,
                            linestyle='dashed'))
ax.text(x_opt + grad_g_opt[0]*scale - 0.35,
        y_opt + grad_g_opt[1]*scale + 0.1,
        r'$\nabla g$', color=TS_AMBER, fontsize=13, fontweight='bold')

# Verify parallel
parallel = np.allclose(grad_f_opt, lam_val * grad_g_opt)
ax.text(0.05, 0.97, r'$\nabla f = \lambda\,\nabla g$ ?' + f'  {parallel} (λ={lam_val})',
        transform=ax.transAxes, fontsize=10,
        color=TS_GREEN if parallel else TS_RED,
        va='top', bbox=dict(facecolor=TS_LIGHT, alpha=0.8))

ax.set_xlim(-1, 5.5); ax.set_ylim(-0.8, 4.0)
ax.set_aspect('equal')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title(r'Lagrange condition: $\nabla f \parallel \nabla g$ at the constrained optimum')
ax.legend(fontsize=9, loc='upper right')
plt.tight_layout()
plt.show()

**What do you see?** At the constrained minimum $(1, 2)$, the level curve of $f$ ($x^2 + y^2 = 5$) is tangent to the constraint line. The gradient arrows $\nabla f$ and $\nabla g$ point in the same direction — the annotation confirms $\nabla f = \lambda \nabla g$ with $\lambda = 2$. Moving along the constraint line in either direction from $(1,2)$ increases $f$ — we are at the minimum. Geometrically, the feasible direction along the line is perpendicular to both gradient arrows (they both point away from the line), so no first-order improvement is possible.

### Section 3 — Lasso vs Ridge: Geometric Sparsity

Visualize why the $\ell_1$ constraint (diamond) produces sparse solutions while the $\ell_2$ constraint (circle) does not. Show the OLS contours expanding to touch each constraint set.

In [ ]:
# ============================================================
# Section 3 — Lasso vs Ridge: geometry of sparsity
# ============================================================
rng = np.random.default_rng(42)
n_obs, p = 40, 2
X = rng.standard_normal((n_obs, p))
# True coefficients: one large, one small
beta_true = np.array([2.5, 0.4])
y_obs = X @ beta_true + rng.standard_normal(n_obs) * 0.6

# OLS solution (unconstrained optimum)
beta_ols = np.linalg.lstsq(X, y_obs, rcond=None)[0]

# --- OLS loss contours ---
b1g = np.linspace(-0.5, 4.0, 300)
b2g = np.linspace(-0.5, 1.5, 300)
B1, B2 = np.meshgrid(b1g, b2g)

R = np.zeros_like(B1)
for i in range(B1.shape[0]):
    for j in range(B1.shape[1]):
        beta_ij = np.array([B1[i,j], B2[i,j]])
        r = y_obs - X @ beta_ij
        R[i,j] = r @ r

# Constraint budgets
t_lasso = 1.2   # ‖β‖₁ ≤ t
t_ridge = 1.2   # ‖β‖₂ ≤ t

# Lasso and Ridge constrained solutions (via scipy)
from scipy.optimize import minimize

def ols_loss(beta):
    r = y_obs - X @ beta
    return r @ r

lasso_sol = minimize(ols_loss, beta_ols * 0.5,
                     constraints={'type': 'ineq',
                                  'fun': lambda b: t_lasso - np.sum(np.abs(b))})
ridge_sol = minimize(ols_loss, beta_ols * 0.5,
                     constraints={'type': 'ineq',
                                  'fun': lambda b: t_ridge**2 - b @ b})

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, sol, t, shape_label, color_cs in [
    (axes[0], lasso_sol.x, t_lasso, r'Lasso: $\|\beta\|_1 \leq t$', TS_AMBER),
    (axes[1], ridge_sol.x, t_ridge, r'Ridge: $\|\beta\|_2 \leq t$', TS_GREEN),
]:
    # OLS contours
    ax.contourf(B1, B2, R, levels=22, cmap='Blues', alpha=0.20)
    cs_lines = ax.contour(B1, B2, R, levels=16,
                          colors=[TS_GRAY], alpha=0.5, linewidths=0.8)

    # Constraint boundary
    theta_c = np.linspace(0, 2*np.pi, 500)
    if 'Lasso' in shape_label:
        # Diamond: |β₁| + |β₂| = t
        cx = t * np.cos(theta_c) / (np.abs(np.cos(theta_c)) + np.abs(np.sin(theta_c)))
        cy = t * np.sin(theta_c) / (np.abs(np.cos(theta_c)) + np.abs(np.sin(theta_c)))
    else:
        # Circle: β₁² + β₂² = t²
        cx = t * np.cos(theta_c)
        cy = t * np.sin(theta_c)

    ax.fill(cx, cy, color=color_cs, alpha=0.18)
    ax.plot(cx, cy, color=color_cs, lw=2.2, label=shape_label)

    # Points
    ax.plot(*beta_ols, '*', color=TS_BLUE, markersize=14, zorder=8,
            label=fr'OLS: $\hat{{\beta}}$ = ({beta_ols[0]:.2f}, {beta_ols[1]:.2f})')
    ax.plot(*sol, 'o', color=TS_RED, markersize=11, zorder=8,
            label=fr'Constrained: ({sol[0]:.3f}, {sol[1]:.3f})')

    # Axes
    ax.axhline(0, color=TS_GRAY, lw=0.8, alpha=0.6)
    ax.axvline(0, color=TS_GRAY, lw=0.8, alpha=0.6)
    ax.set_xlabel(r'$\beta_1$'); ax.set_ylabel(r'$\beta_2$')
    ax.set_title(shape_label)
    ax.legend(fontsize=8, loc='upper right')
    ax.set_xlim(-0.5, 4.0); ax.set_ylim(-0.5, 1.5)
    ax.set_aspect('equal')

    sparse = np.abs(sol[1]) < 0.01
    ax.text(0.05, 0.10,
            fr'$\beta_2 \approx 0$? {sparse}  (β₂ = {sol[1]:.4f})',
            transform=ax.transAxes, fontsize=9,
            color=TS_GREEN if sparse else TS_GRAY,
            bbox=dict(facecolor=TS_LIGHT, alpha=0.8))

plt.suptitle('Lasso (diamond) vs Ridge (circle): geometry of sparsity',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

**What do you see?**

- **Lasso (left)**: The OLS ellipse first contacts the diamond at or near a corner on the $\beta_1$ axis, setting $\beta_2 \approx 0$ exactly. The small true coefficient $\beta_2 = 0.4$ is completely eliminated — Lasso has identified it as negligible relative to the constraint budget.
- **Ridge (right)**: The OLS ellipse contacts the circle smoothly — nowhere near an axis. Both coefficients are shrunk but neither is set to zero. Ridge cannot produce exact sparsity by geometry.
- This is the entire argument for why Lasso does feature selection and Ridge does not. No distributional assumptions are needed — it follows purely from the shape of the constraint set and the expansion of the OLS contours.

**Try modifying**: increase `t_lasso` toward `np.linalg.norm(beta_ols, 1)`. As the budget grows, the Lasso solution moves toward the OLS optimum and sparsity is lost. Decrease it below 0.5 — both coefficients approach zero.

In [ ]:
# Extension workspace
# Suggestions:
# 1. Maximum entropy with a mean constraint: maximize H(p) = -Σp_i log p_i
#    subject to Σp_i = 1 and Σi·p_i = μ for n=5 outcomes.
#    Use SymPy to derive the solution — you should get a Boltzmann distribution.
#
# 2. Elastic Net: the penalty λ₁‖β‖₁ + λ₂‖β‖₂² combines Lasso and Ridge.
#    Plot its constraint set for several (λ₁, λ₂) ratios.
#    When does it look more like a diamond? More like a circle?
#
# 3. Verify the KKT conditions numerically for the Lasso solution:
#    For each coordinate j, check that:
#      −2(Xᵀ(y−Xβ))_j = λ·sign(β_j)  if β_j ≠ 0
#      |−2(Xᵀ(y−Xβ))_j| ≤ λ          if β_j = 0


---

## Summary

| Concept | Statement |
|---|---|
| Lagrange condition | $\nabla f(\boldsymbol{x}^*) = \lambda\,\nabla g(\boldsymbol{x}^*)$ at constrained extremum |
| Lagrangian | $\mathcal{L}(\boldsymbol{x}, \lambda) = f(\boldsymbol{x}) - \lambda\,g(\boldsymbol{x})$ |
| Lagrange equations | $\nabla_{\boldsymbol{x}}\mathcal{L} = \boldsymbol{0}$ and $g(\boldsymbol{x}) = 0$ |
| KKT stationarity | $\nabla f - \boldsymbol{J}_{\boldsymbol{g}}^{\top}\boldsymbol{\lambda} - \boldsymbol{J}_{\boldsymbol{h}}^{\top}\boldsymbol{\mu} = \boldsymbol{0}$ |
| KKT complementary slackness | $\mu_k h_k(\boldsymbol{x}) = 0$, $\mu_k \geq 0$ |
| Max entropy (normalization only) | Uniform distribution $p_i = 1/n$ |
| Lasso constraint | $\min \|\boldsymbol{y} - \boldsymbol{X}\boldsymbol{\beta}\|^2$ s.t. $\|\boldsymbol{\beta}\|_1 \leq t$ |
| Why Lasso is sparse | Diamond constraint has corners on axes; OLS ellipse contacts there |
| Why Ridge is not sparse | Circular constraint is smooth; contact not at corners |

---

## Module 04 Complete

You now have the full optimization toolkit:

| Unit | Core object | Key result |
|---|---|---|
| 01 — Critical Points | $\nabla f = \boldsymbol{0}$, Hessian | Second-derivative test; MLE confirmed as maximum |
| 02 — Hessian & Convexity | PD Hessian, convex functions | Ridge closed form; Newton's method |
| 03 — Lagrange Multipliers | Lagrangian, KKT conditions | Max entropy uniform distribution; Lasso sparsity |

**Up next:** Module 05 — Multiple Integrals

We move from differentiation to integration in multiple variables — iterated integrals, change of variables with the Jacobian determinant, and probability densities over $\mathbb{R}^n$.

---
*Module 04, Unit 03 — Threat Surfaces | fischer³ Education*